# Bronze Layer - Source Exploration

Goal: profile the 3 data sources before writing the ingestion pipeline.

## Sources
- OFS Unemployment BIT (CSV): monthly data by large region, since 2002
- SNB CPI (API JSON): consumer price index, monthly, national
- SNB Policy Rate (API JSON): SNB policy rate, monthly

## Questions to answer
1. What is the actual structure of each source?
2. Is the VALUE column in the unemployment dataset populated?
3. What is the JSON structure returned by the SNB API?
4. Are there missing values or anomalies?

In [1]:
import pandas as pd
import requests
import json

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)

## Source 1 - OFS Unemployment BIT (CSV)

In [2]:
# Load OFS unemployment CSV directly from URL
url_unemployment = "https://dam-api.bfs.admin.ch/hub/api/dam/assets/36519062/master"

df_unemployment = pd.read_csv(url_unemployment, sep=",", encoding="utf-8")

print(f"Shape: {df_unemployment.shape}")
print(f"\nColumns: {df_unemployment.columns.tolist()}")
print(f"\nDtypes:\n{df_unemployment.dtypes}")

Shape: (9888, 14)

Columns: ['INDICATORS_HRCHY', 'INDICATORS_FR', 'INDICATORS_DE', 'GENDER_FR', 'GENDER_DE', 'DETAILS_FR', 'DETAILS_DE', 'PERIOD', 'FREQ', 'MEASURE_FR', 'MEASURE_DE', 'VALUE', 'STATUS', 'STATUS_1']

Dtypes:
INDICATORS_HRCHY     object
INDICATORS_FR        object
INDICATORS_DE        object
GENDER_FR            object
GENDER_DE            object
DETAILS_FR           object
DETAILS_DE           object
PERIOD               object
FREQ                 object
MEASURE_FR           object
MEASURE_DE           object
VALUE               float64
STATUS               object
STATUS_1             object
dtype: object


In [3]:
# Profile key columns
print("=== VALUE column ===")
print(f"Null count: {df_unemployment['VALUE'].isna().sum()}")
print(f"Min: {df_unemployment['VALUE'].min()}")
print(f"Max: {df_unemployment['VALUE'].max()}")

print("\n=== PERIOD sample ===")
print(df_unemployment['PERIOD'].unique()[:5])

print("\n=== GENDER_FR unique values ===")
print(df_unemployment['GENDER_FR'].unique())

print("\n=== DETAILS_FR unique values (regions) ===")
print(df_unemployment['DETAILS_FR'].unique())

print("\n=== INDICATORS_FR unique values ===")
print(df_unemployment['INDICATORS_FR'].unique())

print("\n=== FREQ unique values ===")
print(df_unemployment['FREQ'].unique())

=== VALUE column ===
Null count: 2864
Min: 1.17183824537309
Max: 12.9175566622998

=== PERIOD sample ===
['2002-M01' '2002-M02' '2002-M03' '2002-M04' '2002-M05']

=== GENDER_FR unique values ===
['Total' 'Hommes' 'Femmes']

=== DETAILS_FR unique values (regions) ===
['Total' 'Région lémanique' 'Espace Mittelland' 'Suisse du Nord-Ouest'
 'Zurich' 'Suisse orientale' 'Suisse centrale' 'Tessin']

=== INDICATORS_FR unique values ===
['TOTAL' 'NUTS2']

=== FREQ unique values ===
['M' 'Q' 'Y']


In [4]:
# Investigate nulls by frequency
print("=== Null VALUE count by FREQ ===")
print(df_unemployment.groupby('FREQ')['VALUE'].apply(lambda x: x.isna().sum()))

print("\n=== Total rows by FREQ ===")
print(df_unemployment['FREQ'].value_counts())

# Check what a monthly row with null looks like
print("\n=== Sample null rows ===")
print(df_unemployment[df_unemployment['VALUE'].isna()].head(3))

=== Null VALUE count by FREQ ===
FREQ
M    2304
Q     448
Y     112
Name: VALUE, dtype: int64

=== Total rows by FREQ ===
FREQ
M    6984
Q    2328
Y     576
Name: count, dtype: int64

=== Sample null rows ===
  INDICATORS_HRCHY INDICATORS_FR INDICATORS_DE GENDER_FR GENDER_DE  \
0                P         TOTAL         TOTAL     Total     Total   
1              P_1         NUTS2         NUTS2     Total     Total   
2              P_2         NUTS2         NUTS2     Total     Total   

          DETAILS_FR         DETAILS_DE    PERIOD FREQ           MEASURE_FR  \
0              Total              Total  2002-M01    M  Moyennes mensuelles   
1   Région lémanique    Genferseeregion  2002-M01    M  Moyennes mensuelles   
2  Espace Mittelland  Espace Mittelland  2002-M01    M  Moyennes mensuelles   

                      MEASURE_DE  VALUE STATUS STATUS_1  
0  Durchschnittliche Monatswerte    NaN      O      NaN  
1  Durchschnittliche Monatswerte    NaN      O      NaN  
2  Durchschnittlich

In [5]:
# Check which periods have nulls in monthly data only
df_monthly = df_unemployment[df_unemployment['FREQ'] == 'M']

null_by_period = df_monthly[df_monthly['VALUE'].isna()]['PERIOD'].value_counts().sort_index()
print("=== Periods with null values (monthly) ===")
print(null_by_period)

# Check if TOTAL rows (national aggregate) are always null
print("\n=== Null rate by INDICATORS_FR ===")
print(df_monthly.groupby('INDICATORS_FR')['VALUE'].apply(lambda x: x.isna().sum()))

=== Periods with null values (monthly) ===
PERIOD
2002-M01    24
2002-M02    24
2002-M03    24
2002-M04    24
2002-M05    24
            ..
2009-M08    24
2009-M09    24
2009-M10    24
2009-M11    24
2009-M12    24
Name: count, Length: 96, dtype: int64

=== Null rate by INDICATORS_FR ===
INDICATORS_FR
NUTS2    2016
TOTAL     288
Name: VALUE, dtype: int64


## Source 1 - Findings summary

- 9,888 rows × 14 columns, frequencies M/Q/Y mixed in same file
- VALUE populated from 2010 onwards only (2002-2009 = all null)
- Silver filters: FREQ=M, PERIOD>=2010-M01, GENDER_FR=Total, INDICATORS_FR=NUTS2
- PERIOD format needs parsing: 2002-M01 → 2002-01
- Bilingual columns (FR/DE): drop DE columns in Silver

## Source 2 - SNB CPI (API JSON)

In [6]:
# Call SNB CPI API - full dataset
url_cpi = "https://data.snb.ch/api/cube/plkopr/data/json/fr"

response_cpi = requests.get(url_cpi)
print(f"Status code: {response_cpi.status_code}")
print(f"Response size: {len(response_cpi.content)} bytes")
print(f"\nRaw JSON structure (first 500 chars):")
print(response_cpi.text[:500])

Status code: 200
Response size: 103434 bytes

Raw JSON structure (first 500 chars):
{"timeseries":[{"header":[{"dim":"Vue d’ensemble","dimItem":"Indice suisse – Décembre 2025 = 100"}],"metadata":{"key":"EPB@SNB.plkopr{LD2010100}","frequency":"P1M","scale":""},"values":[{"date":"1921-01","value":19.588398},{"date":"1921-02","value":19.361351},{"date":"1921-03","value":19.098022},{"date":"1921-04","value":18.907351},{"date":"1921-05","value":18.462328},{"date":"1921-06","value":18.117176},{"date":"1921-07","value":17.935669},{"date":"1921-08","value":17.853847},{"date":"1921-09",


In [7]:
data_cpi = response_cpi.json()

print(f"Number of timeseries: {len(data_cpi['timeseries'])}")

for i, ts in enumerate(data_cpi['timeseries']):
    print(f"\n--- Timeseries {i} ---")
    print(f"Header: {ts['header']}")
    print(f"Metadata key: {ts['metadata']['key']}")
    print(f"Number of values: {len(ts['values'])}")
    print(f"First value: {ts['values'][0]}")
    print(f"Last value: {ts['values'][-1]}")

Number of timeseries: 2

--- Timeseries 0 ---
Header: [{'dim': 'Vue d’ensemble', 'dimItem': 'Indice suisse – Décembre 2025 = 100'}]
Metadata key: EPB@SNB.plkopr{LD2010100}
Number of values: 1265
First value: {'date': '1921-01', 'value': 19.588398}
Last value: {'date': '2026-05', 'value': 101.2587}

--- Timeseries 1 ---
Header: [{'dim': 'Vue d’ensemble', 'dimItem': 'Variation en % par rapport au mois correspondant de l’année précédente'}]
Metadata key: EPB@SNB.plkopr{VVP}
Number of values: 1253
First value: {'date': '1922-01', 'value': -15.067669137619117}
Last value: {'date': '2026-05', 'value': 0.6158652370007998}


In [8]:
# Flatten both timeseries into a single DataFrame
dfs = []
for ts in data_cpi['timeseries']:
    series_name = ts['metadata']['key'].split('{')[1].replace('}', '')
    df_ts = pd.DataFrame(ts['values'])
    df_ts['series'] = series_name
    dfs.append(df_ts)

df_cpi = pd.concat(dfs, ignore_index=True)

print(f"Shape: {df_cpi.shape}")
print(f"\nSample:\n{df_cpi[df_cpi['date'] >= '2010-01'].head(6)}")
print(f"\nNull count:\n{df_cpi.isnull().sum()}")

Shape: (2518, 3)

Sample:
         date    value     series
1068  2010-01  94.6843  LD2010100
1069  2010-02  94.8221  LD2010100
1070  2010-03  94.9521  LD2010100
1071  2010-04  95.7637  LD2010100
1072  2010-05  95.6647  LD2010100
1073  2010-06  95.2483  LD2010100

Null count:
date      0
value     0
series    0
dtype: int64


## Source 2 - Findings summary

- 2 timeseries: CPI index (LD2010100) and YoY inflation rate (VVP)
- Date format YYYY-MM - clean, no parsing needed
- Zero nulls
- Data starts 1921, filter to 2010+ in Silver
- Gold KPI: VVP (YoY inflation rate)

## Source 3 - SNB Policy Rate (API JSON)

In [9]:
# Call SNB Policy Rate API
url_policy = "https://data.snb.ch/api/cube/snboffzisa/data/json/fr"

response_policy = requests.get(url_policy)
print(f"Status code: {response_policy.status_code}")
print(f"Number of timeseries: {len(response_policy.json()['timeseries'])}")

for i, ts in enumerate(response_policy.json()['timeseries']):
    print(f"\n--- Timeseries {i} ---")
    print(f"Header: {ts['header']}")
    print(f"Values count: {len(ts['values'])}")
    print(f"First: {ts['values'][0]} | Last: {ts['values'][-1]}")

Status code: 200
Number of timeseries: 10

--- Timeseries 0 ---
Header: [{'dim': 'Vue d’ensemble', 'dimItem': 'Suisse - Taux directeur de la BNS'}]
Values count: 84
First: {'date': '2019-06', 'value': -0.75} | Last: {'date': '2026-05', 'value': 0.0}

--- Timeseries 1 ---
Header: [{'dim': 'Vue d’ensemble', 'dimItem': 'Suisse - BNS - Marge de fluctuation du Libor à trois mois pour le franc - limite inférieure'}]
Values count: 233
First: {'date': '2000-01', 'value': 1.25} | Last: {'date': '2019-05', 'value': -1.25}

--- Timeseries 2 ---
Header: [{'dim': 'Vue d’ensemble', 'dimItem': 'Suisse - BNS - Marge de fluctuation du Libor à trois mois pour le franc - limite supérieure'}]
Values count: 233
First: {'date': '2000-01', 'value': 2.25} | Last: {'date': '2019-05', 'value': -0.25}

--- Timeseries 3 ---
Header: [{'dim': 'Vue d’ensemble', 'dimItem': 'États-Unis - Fed - Marge de fluctuation - limite inférieure'}]
Values count: 317
First: {'date': '2000-01', 'value': 5.5} | Last: {'date': '2026-

## Source 3 - Findings summary

- 10 timeseries returned: SNB policy rate + LIBOR range + other central banks (Fed, ECB, Japan)
- SNB policy rate (TS0): 2019-06 to 2026-05 only (84 values)
- LIBOR range lower/upper (TS1, TS2): 2000-01 to 2019-05 (233 values each)
- Silver reconstruction: midpoint(LIBOR lower + upper) for 2010-2019, SNB rate for 2019+
- Other central banks (TS3-9): dropped in Silver
- Zero nulls expected, date format YYYY-MM consistent with CPI source


## Overall findings
- All 3 sources accessible programmatically (no scraping needed)
- Common date format after Silver parsing: YYYY-MM
- Common time range: 2010-01 to 2026-05
- Grain: monthly for all 3 sources after filtering